# Cost-Aware Simulation Based Inference

**References:**
- Bharti et al. (2025) — [Cost-aware simulation-based inference](https://arxiv.org/abs/2410.07930)

Generating samples for simulation-based inference can have a high computational cost. The cost of each individual simulation often depends on on the parameter value used for simulation. Cost-aware SBI uses importance sampling to encourage sampling from the computationally cheaper parameterisations of the model. In this way it can significantly reduce simulation costs, without any changes to the simulator itself. 

## 1. Imports

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import pickle
import time
import numpy as np

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import bayesflow as bf

from cost_interp_model import CostInterpModel
from cost_plots import *


## 2. Generate Synthetic Cost Data

In a real scenario, cost can be any value that is meaningful for the simulator, for example wall-clock time of the simulation. In this toy example, we use a function of the SIR parameters ($\beta ,\gamma$) to mimic the cost behaviour of the model. This can be any model which 

In [ ]:
def make_synthetic_cost_data(cost_data_dir="cost_data_sir"):
    # Initialize the base SIR simulator
    sir_sim = bf.simulators.benchmark_simulators.SIR()

    os.makedirs(cost_data_dir, exist_ok=True)

    print("Generating synthetic cost data...")
    num_cost_samples = 20
    for i in range(num_cost_samples):
        theta = sir_sim.prior()
        # Simulate cost: higher beta -> higher cost
        actual_cost = np.exp(-theta[0])
        #3 - np.abs(theta[1]) + np.random.normal(0, 0.1)

        with open(f"{cost_data_dir}/sample_{i}.pkl", "wb") as f:
            pickle.dump({"theta": theta, "cost": actual_cost}, f)
            
    return sir_sim


cost_data_dir = "cost_data_sir"
sir_sim = make_synthetic_cost_data(cost_data_dir)


## 3. Fit Cost Interpolation Model

We use a small test Gaussian Process (GP) called `CostInterpModel` to predict the cost given the parameters $\theta$.
This is a user input so can be very varied. 

In [ ]:
print("Fitting cost interpolation model...")
cost_model = CostInterpModel(root=cost_data_dir).fit(length_scale=3.0)


We can visualise how the cost changes with the parameters $\beta$ and $\gamma$. 

In [ ]:
fig = plot_cost_landscape(cost_model)
plt.show(fig)

## 4. Initialize Cost-Aware Simulator

The `CostAwareSimulator` wraps the base simulator and uses the fitted cost model to filter samples based on a regularised cost function $g(c(\theta))$.

In [77]:
cost_aware_sim = bf.simulators.CostAwareSimulator(simulator=sir_sim, cost_model=cost_model)


## 5. Cost-Aware Sampling

We use rejection sampling to obtain a set of cost-efficient parameters.

In [ ]:
print("\nTesting cost-aware sampling...")
num_samples = 1000

#Use the cost-aware sampling method
accepted_samples = cost_aware_sim.sample(batch_shape=(num_samples,1))
accepted_theta = accepted_samples.get("parameters")

#may return more than the requested number of samples. This is because of the way that rejection_sample works, in batches until it reaches the total
#number of samples
print(f"Successfully sampled {len(accepted_theta)} cost-efficient parameters.")


The samples generated by cost-aware sampling can be compared with those taken directly from the prior. 

In [ ]:

plot_prior_samples(cost_model,sir_sim,accepted_theta)

In [ ]:
plot_histograms(sir_sim, accepted_theta)

## 7. Performance Metrics

We evaluate the effectiveness of the cost-aware sampling using Effective Sample Size (ESS) and Computational Gain (CG) on a test batch.

In [ ]:
metrics = cost_aware_sim.compute_metrics({"theta": accepted_theta})
print("\nPerformance Metrics:")
print(f"  ESS: {metrics['ess']:.2f}")
print(f"  CG:  {metrics['cg']:.2f}")


## 8. Running the Expensive Simulator

Simulations are only generated for the accepted theta values.

In [ ]:
results = np.array([sir_sim.observation_model(t) for t in accepted_theta])
print(f"\nSuccessfully simulated {len(results)} samples using the expensive simulator.")


## 9. Training an Approximator on Cost-Efficient Samples

Finally, we show how to use these samples to train a model, using importance weights to correct the sampling bias introduced by cost-filtering.

In [ ]:
print("\nTraining an example model on cost-efficient samples...")
num_accepted = len(accepted_theta)

if num_accepted > 0:
    train_num_candidates = 500
    
    train_observations = [sir_sim.observation_model(t) for t in accepted_theta]
    
    # Load and include dummy data from cost_data_dir
    dummy_files = [f for f in os.listdir(cost_data_dir) if f.endswith('.pkl')]
    for f_name in dummy_files:
        with open(os.path.join(cost_data_dir, f_name), 'rb') as f:
            data = pickle.load(f)
            # In this dummy case, we use the theta as a proxy for observation
            # or we could run the simulator on it
            train_observations.append(sir_sim.observation_model(data['theta']))
            
    train_observations = np.array(train_observations)

    adapter = bf.adapters.Adapter().create_default(inference_variables=["theta"])

    # Compute weights for all samples (accepted + dummy)
    # First, get the parameters for dummy data
    dummy_thetas = []
    dummy_files = [f for f in os.listdir(cost_data_dir) if f.endswith('.pkl')]
    for f_name in dummy_files:
        with open(os.path.join(cost_data_dir, f_name), 'rb') as f:
            dummy_thetas.append(pickle.load(f)['theta'])
    
    all_theta = np.concatenate([accepted_theta, np.array(dummy_thetas)], axis=0)
    train_weights = cost_aware_sim.compute_weights(all_theta)

    train_data = {
        "theta": all_theta,
        "observations": train_observations,
        "weights": train_weights,
    }

    transformed_data = adapter(train_data)


    approximator = bf.approximators.ContinuousApproximator(
        inference_network=inference_net, adapter=adapter
    )

    data_shapes = {
        "inference_variables": accepted_theta.shape[1:],
        "inference_conditions": train_observations.shape[1:],
    }

    approximator.build(data_shapes)

    print(
        f"Model built successfully. Training on {len(all_theta)} samples with importance weights."
    )
    print("The workflow is now complete: Cost-aware sampling -> Model training.")
else:
    print("Skipping model training as no samples were accepted.")



Training an example model on cost-efficient samples...


ValueError: Input 0 with name 'None' of layer 'dense_1' is incompatible with the layer: expected min_ndim=2, found ndim=1. Full shape received: (256,)

## 10. Setting up a workflow

This is a simple set up of a workflow. The cost aware simulator simply replaces the default simulator, to skew the distribution to cheaper sampling regions. 


In [ ]:

inference_net = bf.networks.PointNetwork(points="mean")

workflow = bf.BasicWorkflow(
    simulator=cost_aware_sim,
    adapter=adapter,
    inference_network=inference_net,
)



## 10. Multiple Importance sampling 
When the true posterior lies in the computationally costly region, choosing a single penalty function based on CG and ESS alone may lead to sub-optimal results. This is why we may want to consider multiple cost-aware priors


In [ ]:
regularisation_powers = [0,1,2,3]
for k in regularisation_powers:
    print(k)